In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns; sns.set()

import statsmodels.api as sm
import statsmodels.formula.api as smf

from scipy.stats import t, f, norm

In [113]:
df = pd.read_csv('/content/Newest_with_headers_OCACS_2021_Housing_Characteristics_for_ZIP_Code_Tabulation_Areas copy.csv', header=1)
df.dtypes
df.head(5)

,FID,ZCTA5CE20,GEOID20,CLASSFP20,MTFCC20,FUNCSTAT20,ALAND20,AWATER20,INTPTLAT20,INTPTLON20,...,H23: 15.0 to 19.9 percent,H23: 20.0 to 24.9 percent,H23: 25.0 to 29.9 percent,H23: 30.0 to 34.9 percent,H23: 35.0 to 39.9 percent,H23: 40.0 to 49.9 percent,H23: 50.0 percent or more,H23: Not computed,Shape__Area,Shape__Length
0,1,92657,92657,B5,G6350,S,21494838,1549909,33.595058,-117.829131,...,19,151,121,46,70,49,141,57,0.002238,0.285264
1,2,92704,92704,B5,G6350,S,19184426,62530,33.721131,-117.905110,...,1157,1051,1170,860,688,1025,2273,182,0.001872,0.295664
2,3,92701,92701,B5,G6350,S,8351656,0,33.748622,-117.858714,...,789,1043,885,650,655,1301,2480,259,0.000813,0.186110
3,4,92697,92697,B5,G6350,S,421086,62414,33.649727,-117.849085,...,0,0,0,0,0,0,0,0,0.000047,0.046563
4,5,92602,92602,B5,G6350,S,39127972,237832,33.736498,-117.734933,...,497,699,910,382,374,385,1411,173,0.003829,0.403354


In [81]:
drop_cols = [
    'City/Neighborhood',
    'Month',
    'Year',
    'Distance to Nearest Beach (Km)',
    'Distance from Disneyland (km)',
    "Distance from Knott's Berry Farm (km)",
    "Knott's Berry Farm historical attendance data",
    "Disneyland Resort historical attendance data"

]
beach_cols = [col for col in df.columns if "Nearest Beach Name" in col]
drop_cols.extend(beach_cols)
df_dropped_uncleaned = df.drop(columns=drop_cols)
df_dropped_uncleaned.head(5)

,City,Zipcode,Date,Price Index,Median Household Income Last_12,City Park Scores,Crime Data City Level (Arrest Disposition),Distance from Irvine Spectrum (km)
0,Aliso Viejo,92656,2014-01-31,486155.5583,97807.0,1.95,309.0,8.139577
1,Aliso Viejo,92656,2014-02-28,489076.6266,97807.0,1.95,309.0,8.139577
2,Aliso Viejo,92656,2014-03-31,492136.1673,97807.0,1.95,309.0,8.139577
3,Aliso Viejo,92656,2014-04-30,495451.7403,97807.0,1.95,309.0,8.139577
4,Aliso Viejo,92656,2014-05-31,498080.3425,97807.0,1.95,309.0,8.139577


In [83]:
df_dropped_uncleaned.shape

(20777, 8)

In [97]:
# look for missing & wrong obj types
print(df_dropped_uncleaned.dtypes)
df_dropped_uncleaned['City'].unique()
df_dropped_uncleaned['Zipcode'].unique()
df_dropped_uncleaned['Date'].unique()
df_dropped_uncleaned['Price Index'].unique()
df_dropped_uncleaned['Median Household Income Last_12'].unique()
df_dropped_uncleaned['City Park Scores'].unique()
df_dropped_uncleaned['Crime Data City Level (Arrest Disposition)'].unique()
df_dropped_uncleaned['Distance from Irvine Spectrum (km)'].unique()

print(df_dropped_uncleaned.isna().sum())

# df['Platform'].unique()
# df['Year'].unique()  # missing value 'unknown'
# df['Genre'].unique()
# df['NA_Sales'].unique()
# df['EU_Sales'].unique()  # missing value 'unknown', object
# df['JP_Sales'].unique()
# df['Other_Sales'].unique()

City                                           object
Zipcode                                         int64
Date                                           object
Price Index                                   float64
Median Household Income Last_12               float64
City Park Scores                              float64
Crime Data City Level (Arrest Disposition)    float64
Distance from Irvine Spectrum (km)            float64
dtype: object
City                                             0
Zipcode                                          0
Date                                             0
Price Index                                      0
Median Household Income Last_12               2627
City Park Scores                                 0
Crime Data City Level (Arrest Disposition)    1800
Distance from Irvine Spectrum (km)               0
dtype: int64


In [100]:
# remove missing data rows
print(f'before dropping: {df_dropped_uncleaned.shape}')
df_cleaned = df_dropped_uncleaned.dropna()
print(f'after dropping: {df_cleaned.shape}')
print(f'total rows dropped: {df_dropped_uncleaned.shape[0] - df_cleaned.shape[0]}')

before dropping: (20777, 8)
after dropping: (16530, 8)
total rows dropped: 4247


In [101]:
df_cleaned.head(5)

,City,Zipcode,Date,Price Index,Median Household Income Last_12,City Park Scores,Crime Data City Level (Arrest Disposition),Distance from Irvine Spectrum (km)
0,Aliso Viejo,92656,2014-01-31,486155.5583,97807.0,1.95,309.0,8.139577
1,Aliso Viejo,92656,2014-02-28,489076.6266,97807.0,1.95,309.0,8.139577
2,Aliso Viejo,92656,2014-03-31,492136.1673,97807.0,1.95,309.0,8.139577
3,Aliso Viejo,92656,2014-04-30,495451.7403,97807.0,1.95,309.0,8.139577
4,Aliso Viejo,92656,2014-05-31,498080.3425,97807.0,1.95,309.0,8.139577


In [109]:
# Keep only the most recent entry for each Zip Code
df_cleaned = df_cleaned.sort_values('Date')
df_latest = df_cleaned.groupby('Zipcode').tail(1)
df_latest.head(5)
df_latest.shape
# Now you can safely drop the Date
df_final = df_latest.drop(columns=['Date'])
df_final = df_final.sort_values('Median Household Income Last_12', ascending=False)
df_final

,City,Zipcode,Price Index,Median Household Income Last_12,City Park Scores,Crime Data City Level (Arrest Disposition),Distance from Irvine Spectrum (km)
11949,Newport Beach,92657,3.973774e+06,155579.0,5.86,1914.0,12.519541
20524,Villa Park,92861,1.910865e+06,150918.0,0.00,63.0,21.693666
11948,Newport Beach,92660,2.888447e+06,113054.0,5.86,1914.0,15.517666
11947,Newport Beach,92625,3.386157e+06,112727.0,5.86,1914.0,15.642958
1198,Anaheim,92808,7.546620e+05,106892.0,1.50,12231.0,28.694330
8988,Irvine,92602,1.280001e+06,105587.0,18.85,4864.0,9.363520
20764,Yorba Linda,92886,1.190136e+06,105448.0,2.74,773.0,29.097924
11133,Laguna Niguel,92677,1.209597e+06,100480.0,2.54,737.0,13.015319
12681,Rancho Santa Margarita,92688,8.898601e+05,100388.0,0.00,241.0,10.115455
10893,Laguna Beach,92651,2.701512e+06,99893.0,15.52,449.0,12.835238


In [106]:
df_final.to_csv('Orange_County_Home_Prices_Latest.csv', index=False)

In [122]:

df = pd.read_csv('Newest_with_headers_OCACS_2021_Housing_Characteristics_for_ZIP_Code_Tabulation_Areas copy.csv', header=0, skiprows=[1])
df.shape


(90, 418)

In [118]:
refined_map = {
    'ZCTA5CE20': 'Zip_Code',
    'INTPTLAT20': 'Latitude',
    'INTPTLON20': 'Longitude',

    # H02: Structure Density (Excluding mobile homes/boats)
    'H02B25024e4': 'Duplexes',
    'H02B25024e5': 'Small_Apartments_3_to_4',
    'H02B25024e6': 'Mid_Apartments_5_to_9',
    'H02B25024e7': 'Large_Apartments_10_to_19',
    'H02B25024e8': 'Major_Apartments_20_to_49',
    'H02B25024e9': 'High_Rise_50_plus',

    # H08: Renter vs Owner & Population
    'H08B25008e1': 'Total_Pop_in_Units',
    'H08B25008e2': 'Pop_Owners',
    'H08B25008e3': 'Pop_Renters',

    # H16: Property Value Distribution
    'H16B25076e1': 'Home_Value_Lower_Quartile',
    'H16B25077e1': 'Home_Value_Median',
    'H16B25078e1': 'Home_Value_Upper_Quartile',

    # Other Essentials
    'H04B25035e1': 'Median_Year_Built',
    'H07B25010e1': 'Avg_Household_Size'
}


In [134]:
df_refined = df[list(refined_map.keys())].copy()
df_refined = df_refined.rename(columns=refined_map)

df_refined = df_refined.dropna()
df_refined = df_refined[df_refined['Total_Pop_in_Units'] > 0]
df_refined

,Zip_Code,Latitude,Longitude,Detached_Houses,Attached_Houses,Duplexes,Small_Apartments_3_to_4,Mid_Apartments_5_to_9,Large_Apartments_10_to_19,Major_Apartments_20_to_49,High_Rise_50_plus,Total_Pop_in_Units,Pop_Owners,Pop_Renters,Home_Value_Lower_Quartile,Home_Value_Median,Home_Value_Upper_Quartile,Median_Year_Built,Avg_Household_Size
0,92657,33.595058,-117.829131,2820,708,15,45,153,108,213,124,9342,7623,1719,1195300.0,2000001.0,2000001.0,1998.0,2.56
1,92704,33.721131,-117.905110,9857,1431,207,1699,1905,1586,746,1200,79472,40111,39361,380800.0,534800.0,667200.0,1971.0,4.04
2,92701,33.748622,-117.858714,4029,1122,551,930,879,1535,1837,1842,48398,13864,34534,377200.0,521700.0,658800.0,1968.0,3.92
4,92602,33.736498,-117.734933,4653,1672,0,119,765,863,746,1256,27425,11953,15472,767200.0,933600.0,1198100.0,2005.0,3.01
5,92683,33.752429,-117.993872,16075,2383,504,1695,1172,855,887,2175,90959,49492,41467,515900.0,651200.0,810900.0,1969.0,3.29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,92886,33.895411,-117.800829,13238,2420,73,542,285,79,203,345,50437,41302,9135,645100.0,864700.0,1112900.0,1979.0,3.01
86,92808,33.856247,-117.740232,4699,1846,0,313,350,152,146,426,21496,16764,4732,596900.0,807100.0,989300.0,1993.0,2.76
87,92679,33.649438,-117.561405,8716,1564,99,184,160,129,32,29,31353,28651,2702,765800.0,970200.0,1245800.0,1993.0,2.99
88,92692,33.612175,-117.640452,12790,1991,69,631,506,421,174,1047,46123,35772,10351,623300.0,795500.0,960000.0,1986.0,2.78


In [135]:
rows_with_na = df_refined[df_refined.isna().any(axis=1)]
rows_with_na.head(5)


,Zip_Code,Latitude,Longitude,Detached_Houses,Attached_Houses,Duplexes,Small_Apartments_3_to_4,Mid_Apartments_5_to_9,Large_Apartments_10_to_19,Major_Apartments_20_to_49,High_Rise_50_plus,Total_Pop_in_Units,Pop_Owners,Pop_Renters,Home_Value_Lower_Quartile,Home_Value_Median,Home_Value_Upper_Quartile,Median_Year_Built,Avg_Household_Size


In [136]:
df_refined.to_csv('OC_Housing_Detailed_Refined.csv', index=False)

In [25]:
df = pd.read_csv('ConsumerData.csv')
df.head(5)

,RecordID,MAK,BaseMak,Address,City,State,Zipcode,Latitude,Longitude,OwnerRenter,...,Gardening,EnvironmentalIssues,HomeImprovement,HomeImprovementDIY,OutdoorsGrouping,InvestmentsForeign,BeautyCosmetics,TVCable,WirelessCellularPhoneOwner,EducationOnline
0,1,9005082639,0,22 San Gabriel,Rancho Santa Margarita,CA,92688,33.643921,-117.622469,O,...,Y,Y,Y,Y,Y,Y,Y,Y,Y,NaN
1,2,5389949962,1465166719,22751 El Prado,Rancho Santa Margarita,CA,92688,33.632943,-117.597520,R,...,Y,NaN,Y,NaN,Y,Y,NaN,Y,NaN,NaN
2,3,6768710419,0,24 Buenaventura,Rancho Santa Margarita,CA,92688,33.645838,-117.621734,O,...,Y,NaN,Y,NaN,Y,Y,NaN,Y,Y,NaN
3,4,2234668744,0,8 San Carlos,Rancho Santa Margarita,CA,92688,33.640690,-117.584853,O,...,Y,Y,Y,Y,Y,Y,Y,Y,Y,NaN
4,5,5631554032,0,1 Calle Horca,Rancho Santa Margarita,CA,92688,33.643760,-117.587714,R,...,NaN,NaN,NaN,NaN,NaN,Y,NaN,Y,Y,NaN


In [21]:
df = pd.read_csv('OC_Housing_Detailed_Cleaned.csv')

In [22]:
# Your target coordinates
target_lat = 33.636287
target_lon = -117.591499

# Calculate simple Euclidean distance (works well for small areas like Orange County)
df['distance'] = np.sqrt((df['Latitude'] - target_lat)**2 + (df['Longitude'] - target_lon)**2)

# Find the closest row
closest_row = df.loc[df['distance'].idxmin()]
print("Closest match:")
print(closest_row)

# Get top 5 closest locations
top_5 = df.nsmallest(5, 'distance')
print("\nTop 5 closest locations:")
top_5.head(5)

Closest match:
Zip_Code                      92688.000000
Latitude                         33.618730
Longitude                      -117.610799
Detached_Houses                7483.000000
Attached_Houses                3790.000000
Duplexes                        148.000000
Small_Apartments_3_to_4         490.000000
Mid_Apartments_5_to_9          1062.000000
Large_Apartments_10_to_19       993.000000
Major_Apartments_20_to_49       301.000000
High_Rise_50_plus              1524.000000
Total_Pop_in_Units            44386.000000
Pop_Owners                    31408.000000
Pop_Renters                   12978.000000
Home_Value_Lower_Quartile    512800.000000
Home_Value_Median            673400.000000
Home_Value_Upper_Quartile    856100.000000
Median_Year_Built              1993.000000
Avg_Household_Size                2.870000
distance                          0.026092
Name: 16, dtype: float64

Top 5 closest locations:


,Zip_Code,Latitude,Longitude,Detached_Houses,Attached_Houses,Duplexes,Small_Apartments_3_to_4,Mid_Apartments_5_to_9,Large_Apartments_10_to_19,Major_Apartments_20_to_49,High_Rise_50_plus,Total_Pop_in_Units,Pop_Owners,Pop_Renters,Home_Value_Lower_Quartile,Home_Value_Median,Home_Value_Upper_Quartile,Median_Year_Built,Avg_Household_Size,distance
16,92688,33.618730,-117.610799,7483,3790,148,490,1062,993,301,1524,44386,31408,12978,512800.0,673400.0,856100.0,1993.0,2.87,0.026092
86,92679,33.649438,-117.561405,8716,1564,99,184,160,129,32,29,31353,28651,2702,765800.0,970200.0,1245800.0,1993.0,2.99,0.032842
87,92692,33.612175,-117.640452,12790,1991,69,631,506,421,174,1047,46123,35772,10351,623300.0,795500.0,960000.0,1986.0,2.78,0.054569
37,92691,33.611945,-117.665867,11817,2159,64,745,870,143,87,1035,47033,35952,11081,577900.0,750800.0,897600.0,1975.0,2.87,0.078250
69,92610,33.683363,-117.654080,3030,612,26,69,180,176,75,254,12524,9581,2943,627200.0,807300.0,928100.0,1995.0,2.96,0.078310


In [15]:
import sqlite3

def csv_to_sqlite(csv_file, db_file, table_name='consumer_data'):
  df = pd.read_csv(csv_file)
  conn = sqlite3.connect(db_file)
  df.to_sql(table_name, conn, if_exists='replace', index=False)
  cursor = conn.cursor()
  index_columns = ['Zipcode', 'Latitude', 'Longitude', 'OwnerRenter', 'MaritalStatus']
  for col in index_columns:
    if col in df.columns:
      try:
        cursor.execute(f"CREATE INDEX idx_{col} ON {table_name} ({col})")
      except sqlite3.OperationalError as e:
        print(f"failed to make index on {col}: {e}")
  conn.close()
  print(f"db created: {db_file}")
  return db_file


In [23]:
csv_to_sqlite('ConsumerData.csv', 'consumer_data.db')

db created: consumer_data.db


'consumer_data.db'

In [17]:
import sqlite3
import pandas as pd

def quick_check(db_file, table_name='consumer_data'):

    conn = sqlite3.connect(db_file)

    cursor = conn.cursor()
    cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
    row_count = cursor.fetchone()[0]

    cursor.execute(f"PRAGMA table_info({table_name})")
    columns = cursor.fetchall()
    col_count = len(columns)

    print(f"Table: {table_name}")
    print(f"Rows: {row_count:,}")
    print(f"Columns: {col_count}")
    print(f"\nFirst 5 rows:")

    # Show first 5 rows
    df = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 5", conn)
    print(df)

    # Show column names
    print(f"\nColumn names:")
    print([col[1] for col in columns])

    conn.close()
    return df

# Run quick check
df_preview = quick_check('consumer_data.db')

Table: consumer_data
Rows: 10,000
Columns: 44

First 5 rows:
   RecordID         MAK     BaseMak          Address                    City  \
0         1  9005082639           0   22 San Gabriel  Rancho Santa Margarita   
1         2  5389949962  1465166719   22751 El Prado  Rancho Santa Margarita   
2         3  6768710419           0  24 Buenaventura  Rancho Santa Margarita   
3         4  2234668744           0     8 San Carlos  Rancho Santa Margarita   
4         5  5631554032           0    1 Calle Horca  Rancho Santa Margarita   

  State  Zipcode   Latitude   Longitude OwnerRenter  ...  Gardening  \
0    CA    92688  33.643921 -117.622469           O  ...          Y   
1    CA    92688  33.632943 -117.597520           R  ...          Y   
2    CA    92688  33.645838 -117.621734           O  ...          Y   
3    CA    92688  33.640690 -117.584853           O  ...          Y   
4    CA    92688  33.643760 -117.587714           R  ...       None   

  EnvironmentalIssues HomeImpro

In [19]:
df.head(5)
df.shape

(10000, 45)

In [20]:
import sqlite3
import pandas as pd

def diagnose_column_mismatch(db_file, csv_file, table_name='consumer_data'):
    """Find why columns are missing in database"""

    # Read CSV
    df_csv = pd.read_csv(csv_file)
    csv_columns = set(df_csv.columns)
    print(f"CSV columns: {len(csv_columns)}")

    # Get database columns
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()
    cursor.execute(f"PRAGMA table_info({table_name})")
    db_columns = set(row[1] for row in cursor.fetchall())
    print(f"Database columns: {len(db_columns)}")

    # Find missing columns
    missing_columns = csv_columns - db_columns
    print(f"\n❌ Missing columns in database: {missing_columns}")

    # Check each missing column
    for col in missing_columns:
        print(f"\n--- Investigating '{col}' ---")

        # Check data type in CSV
        dtype = df_csv[col].dtype
        print(f"  CSV dtype: {dtype}")

        # Check for problematic values
        unique_vals = df_csv[col].dropna().unique()
        print(f"  Unique values: {unique_vals[:10]}")

        # Check if column is all NaN/empty
        null_count = df_csv[col].isna().sum()
        print(f"  Null/Empty count: {null_count}/{len(df_csv)} ({null_count/len(df_csv)*100:.1f}%)")

        # Check for problematic characters in column name
        if any(char in str(col) for char in [' ', '-', '.', '(', ')', '/', '\\']):
            print(f"  ⚠️ Column name has special characters: '{col}'")

        # Check if column might be all empty strings
        if df_csv[col].dtype == 'object':
            empty_strings = (df_csv[col] == '').sum()
            if empty_strings == len(df_csv):
                print(f"  ⚠️ Column contains only empty strings")

    conn.close()
    return missing_columns

# Run diagnosis
missing = diagnose_column_mismatch('consumer_data.db', 'ConsumerData.csv')

CSV columns: 44
Database columns: 44

❌ Missing columns in database: set()
